# Домашняя работа №4

**ФИО**: Дергалов Никита Олегович

**Группа**: ИУ6-55Б

**Вариант**: №2

In [1]:
surname = "Дергалов" #Ваша фамилия

alp = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя'
w = [1, 42, 21, 21, 35,  6, 44, 26, 18, 44, 38, 26, 14, 43,  4, 49, 45,
        7, 42, 29,  4,  9, 36, 34, 31, 29,  5, 30,  4, 19, 28, 25, 33]

d = dict(zip(alp, w))
variant =  sum([d[el] for el in surname.lower()]) % 40 + 1

print("Задача № 1: ", variant % 3 + 1)
print("Задача № 2: ", variant % 2 + 1 )

Задача № 1:  3
Задача № 2:  2


Т.е. в данном дз будет выполняться Задача №2: вариант 2

## Задание:
1) Разделите данные с рейтингами на обучающее (train_init - 0.8) и тестовое подмножества (test - 0.2), определите среднее значение рейтинга в обучающем подмножестве и вычислите rmse для тестового подмножества, если для всех значений из test предсказывается среднее значение рейтинга

2) Реализуйте коллаборативную фильтрацию по схожести объектов. Для определения схожести используйте train_init, для расчета rmse - test

3) Определите rmse для тестового подмножества


Подключим необходимые библиотеки

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

Инициализируем spark-сессию

In [2]:
spark = SparkSession.builder \
    .appName("ItemCF-MovieLens") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

data_path = "ml-latest-small"

ratings = spark.read.csv(
    f"{data_path}/ratings.csv",
    header=True,
    inferSchema=True
).select("userId", "movieId", "rating")  # timestamp не нужен

25/12/10 15:40:27 WARN Utils: Your hostname, Kvasik resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp7s0)
25/12/10 15:40:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/10 15:40:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Шаг 1: бейзлайн по среднему рейтингу и RMSE

Проводим train-test split в отношении (0.8;0.2)

In [3]:
train_init, test = ratings.randomSplit([0.8, 0.2], seed=42)
train_init.cache()
test.cache()

DataFrame[userId: int, movieId: int, rating: double]

Находим среднее значение рейтинга в тестовой выборке

In [4]:
global_mean = train_init.agg(F.avg("rating").alias("mean_rating")).first()["mean_rating"]
global_mean

3.5039340762987417

Предсказываем для всех пар (user, movie) из test одно и то же значение global_mean и считаем RMSE:

In [5]:
test_mean_pred = test.withColumn("prediction", F.lit(float(global_mean)).cast(DoubleType()))

evaluator = RegressionEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="rmse"
)

rmse_baseline = evaluator.evaluate(test_mean_pred)
print("Baseline RMSE (global mean):", rmse_baseline)


Baseline RMSE (global mean): 1.050437770716982


### Шаг 2: вычисление похожести объектов (item-based CF)

Для item-based CF считаем похожесть фильмов по ко‑оценкам одних и тех же пользователей. Можно использовать косинусную меру сходства на векторе рейтингов.

Строим все пары фильмов, которые хотябы один пользователь оценил вместе

In [9]:
ratings1 = train_init.alias("r1")
ratings2 = train_init.alias("r2")

pairs = ratings1.join(
    ratings2,
    (F.col("r1.userId") == F.col("r2.userId")) & (F.col("r1.movieId") < F.col("r2.movieId"))
)

pairs.head(5)

[Row(userId=1, movieId=1, rating=4.0, userId=1, movieId=5060, rating=5.0),
 Row(userId=1, movieId=1, rating=4.0, userId=1, movieId=4006, rating=4.0),
 Row(userId=1, movieId=1, rating=4.0, userId=1, movieId=3809, rating=4.0),
 Row(userId=1, movieId=1, rating=4.0, userId=1, movieId=3793, rating=5.0),
 Row(userId=1, movieId=1, rating=4.0, userId=1, movieId=3744, rating=4.0)]

Агрегируем по парам фильмов статистики для косинуса.

In [10]:
# агрегируем по (movieId1, movieId2)
item_pairs_stats = pairs.groupBy(
    F.col("r1.movieId").alias("movieId1"),
    F.col("r2.movieId").alias("movieId2")
).agg(
    F.count("*").alias("n_common"),
    F.sum(F.col("r1.rating") * F.col("r2.rating")).alias("sum_xy"),
    F.sum(F.col("r1.rating") * F.col("r1.rating")).alias("sum_xx"),
    F.sum(F.col("r2.rating") * F.col("r2.rating")).alias("sum_yy")
)

item_pairs_stats.head(5)

[Row(movieId1=1, movieId2=1348, n_common=6, sum_xy=79.25, sum_xx=104.25, sum_yy=67.25),
 Row(movieId1=1, movieId2=1208, n_common=45, sum_xy=704.75, sum_xx=670.0, sum_yy=789.5),
 Row(movieId1=3, movieId2=367, n_common=20, sum_xy=213.0, sum_xx=224.25, sum_yy=227.25),
 Row(movieId1=110, movieId2=1298, n_common=9, sum_xy=144.75, sum_xx=160.25, sum_yy=138.25),
 Row(movieId1=110, movieId2=736, n_common=45, sum_xy=623.75, sum_xx=803.0, sum_yy=527.0)]

Считаем косинусное сходство и отфильтровываем пары с малым числом общих пользователей.

In [11]:
# косинусное сходство
item_sims = item_pairs_stats \
    .withColumn(
        "similarity",
        F.col("sum_xy") / (F.sqrt(F.col("sum_xx")) * F.sqrt(F.col("sum_yy")))
    )

# фильтруем пары с небольшим количеством общих пользователей
min_common_users = 3
item_sims = item_sims.filter(F.col("n_common") >= min_common_users)

item_sims.cache()
item_sims.head(5)

[Row(movieId1=1, movieId2=1348, n_common=6, sum_xy=79.25, sum_xx=104.25, sum_yy=67.25, similarity=0.9464879620255023),
 Row(movieId1=1, movieId2=1208, n_common=45, sum_xy=704.75, sum_xx=670.0, sum_yy=789.5, similarity=0.9689951783118324),
 Row(movieId1=3, movieId2=367, n_common=20, sum_xy=213.0, sum_xx=224.25, sum_yy=227.25, similarity=0.9435424234354571),
 Row(movieId1=110, movieId2=1298, n_common=9, sum_xy=144.75, sum_xx=160.25, sum_yy=138.25, similarity=0.9724941181831873),
 Row(movieId1=110, movieId2=736, n_common=45, sum_xy=623.75, sum_xx=803.0, sum_yy=527.0, similarity=0.9588429477649951)]

Полученная таблица item_sims содержит пары фильмов и их похожесть, рассчитанную по ratings из train_init

### Шаг 3: предсказания по item-based CF и RMSE

Предсказание рейтинга пользователя $u$ к фильму $i$ строим как взвешенное среднее его рейтингов на похожих фильмах:
$$
\hat{r}_{u,i}=\frac{∑_{j∈N_u(i)}sim(i,j) r_{u,j}}{∑_{j∈N_u(i)}∣sim(i,j)∣}
$$

где $N_u(i)$ — фильмы $j$, которые пользователь $u$ оценил в train_init и для которых есть похожесть с фильмом $i$.

Берем все оценки пользователя $u$ из train_init.

In [12]:
K = 50  # количество наиболее похожих фильмов, используемых в предсказании

# рейтинги пользователя из train_init
user_train_ratings = train_init.select(
    F.col("userId").alias("u"),
    F.col("movieId").alias("j"),
    F.col("rating").alias("r_u_j")
)
user_train_ratings.head(5)

[Row(u=1, j=1, r_u_j=4.0),
 Row(u=1, j=3, r_u_j=4.0),
 Row(u=1, j=47, r_u_j=5.0),
 Row(u=1, j=50, r_u_j=5.0),
 Row(u=1, j=70, r_u_j=3.0)]

Находим для целевого фильма $i$ похожие элементы.

In [13]:
# для пары (u, i) из теста ищем фильмы j, которые тот же пользователь оценил, и похожесть (i, j)
test_ui = test.select(
    F.col("userId").alias("u"),
    F.col("movieId").alias("i"),
    F.col("rating").alias("true_rating")
)

# соединяем тест с похожестями: нам нужны строки, где i совпадает с movieId1 или movieId2
sim_i_j_1 = test_ui.join(
    item_sims,
    test_ui.i == item_sims.movieId1,
    how="left"
).select(
    "u", "i",
    F.col("movieId2").alias("j"),
    "true_rating",
    "similarity"
)

sim_i_j_2 = test_ui.join(
    item_sims,
    test_ui.i == item_sims.movieId2,
    how="left"
).select(
    "u", "i",
    F.col("movieId1").alias("j"),
    "true_rating",
    "similarity"
)

sim_i_j = sim_i_j_1.unionByName(sim_i_j_2).filter(F.col("j").isNotNull())

# теперь добавляем рейтинг пользователя на j из train_init
sim_with_ratings = sim_i_j.join(
    user_train_ratings,
    on=["u", "j"],
    how="inner"  # только те j, которые пользователь действительно оценил в train_init
)

Оставляем top‑K (50) наиболее похожих и считаем взвешенное среднее.

In [14]:
# выбираем top-K по |similarity| для каждой пары (u, i)
window = Window.partitionBy("u", "i").orderBy(F.desc(F.abs(F.col("similarity"))))
sim_topk = sim_with_ratings.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") <= K)

# считаем взвешенное среднее
predictions_itemcf = sim_topk.groupBy("u", "i", "true_rating").agg(
    (F.sum(F.col("similarity") * F.col("r_u_j")) /
     F.sum(F.abs(F.col("similarity")))).alias("prediction")
)

# есть пользователи/фильмы, для которых не нашлось похожих соседей;
# можно подставить глобальное среднее как fallback
predictions_full = test_ui.join(
    predictions_itemcf.select("u", "i", "prediction"),
    on=["u", "i"],
    how="left"
).withColumn(
    "prediction",
    F.when(F.col("prediction").isNull(), F.lit(float(global_mean))).otherwise(F.col("prediction"))
)

# считаем RMSE для item-based CF
rmse_itemcf = evaluator.evaluate(
    predictions_full.select(
        F.col("true_rating").alias("rating"),
        "prediction"
    )
)

Полученный rmse

In [17]:
print("Item-based CF RMSE:", rmse_itemcf)

Item-based CF RMSE: 0.9655158134838144
